In [1]:
import healpy as hp
from astropy.io import fits
from astropy import conf
from astropy.table import Table, vstack, hstack, join
from astropy.coordinates import Distance
from astropy.cosmology import Planck18, LambdaCDM
import numpy as np
import fitsio
import numpy.lib.recfunctions as rfn
import os
import pickle
import vast.catalog.void_catalog as void_catalog

## Load Galaxies

In [ ]:
#gal_dir = '/global/cfs/cdirs/desi/users/hrincon/DESIVAST/galaxy_catalog/'

# Angular mask of 100% Survey Completeness
#mask_file = f"{gal_dir}/mask/loa_mask.fits"
#smoothed_mask_file = f"{gal_dir}/mask/loa_mask_smoothed.fits"

smoothed_mask_file = '/pscratch/sd/h/hrincon/loa_mask_smoothed.fits'


# Redshift limits
zmin = 0.
zmax = 0.24

# Read in mask
# There are two mask versions. The first ("mask") covers all regions of BGS that have been completed (for up to 4 passes).
# The second ("smoothed_mask") removes small non-contiguous regions from the mask for the purpose of void-finding on a contigious mask
# From now on, we will use "mask" to refer to "smoothed mask"
mask=fitsio.read(smoothed_mask_file)
#smoothed_mask=fitsio.read(smoothed_mask_file)
nside=hp.npix2nside(len(mask)) #healpix nside parameter

def get_spec_file_name (healpix_index):
    # Fastspec fit files are split up into different healpix groups that cover differnt regions of the sky
    # There are 12 files total
    formated_index = str(healpix_index).zfill(2)
    
    file_name = f'/global/cfs/cdirs/desi/vac/dr2/fastspecfit/loa/v1.0/catalogs/fastspec-loa-main-bright-nside1-hp{formated_index}.fits'

    return file_name

# Save information about the mask and masked galaxies


ngc_catalog = Table(names = ['TARGETID','RA','DEC','Z',
                          'ABSMAG01_SDSS_U','ABSMAG01_SDSS_G', 'ABSMAG01_SDSS_R', 'ABSMAG01_SDSS_Z',
                          'LOGMSTAR', 'SFR', 'HALPHA_EW'])

sgc_catalog = Table(names = ['TARGETID','RA','DEC','Z',
                          'ABSMAG01_SDSS_U','ABSMAG01_SDSS_G', 'ABSMAG01_SDSS_R', 'ABSMAG01_SDSS_Z',
                          'LOGMSTAR', 'SFR', 'HALPHA_EW'])

#Open the FastSpecFit VAC
for idx in range (12): #12 bgs files total
    
    with fitsio.FITS(get_spec_file_name(idx)) as full_catalog:

        
        print("reading data", idx)
        metadata = full_catalog[1]['TARGETID','Z','ZWARN','DELTACHI2','SPECTYPE','RA','DEC','BGS_TARGET', 'SURVEY', 'PROGRAM'
                           ][:]
        specphot = full_catalog[2][
                           'ABSMAG01_SDSS_U', 'ABSMAG01_SDSS_G',
                           'ABSMAG01_SDSS_R', 'ABSMAG01_SDSS_Z',
                           'LOGMSTAR', 'SFR', #'HALPHA_FLUX', 'HBETA_FLUX',
                          ][:]

        fastspec = full_catalog[3]['HALPHA_EW',][:]


        catalog = rfn.merge_arrays([metadata, specphot, fastspec], flatten=True, usemask=False)

        del metadata, specphot, fastspec

        print(len(catalog), "target observations read in")
        
        # Select BGS Bright galaxies
        select = np.where(
                   (catalog['SPECTYPE']=='GALAXY') 
                   & (catalog['SURVEY']=='main')
                   & (catalog['PROGRAM']=='bright')
                 )
        
        catalog=catalog[select]
            
        #check for duplicate targets
        _, select = np.unique(catalog['TARGETID'], return_index=True)
        if len(catalog) != len(select):
            raise ValueError(f'Duplicate galaxies detected. {len(select)} out of {len(catalog)} are unique')
                
        print(len(catalog), "bright time galaxies")
    
        # Impose redshift limits
        print("Imposing redshift limits")
        select = np.where((catalog['Z']>zmin)  # > zmin and not >=zmin to avoid galaxies at origin
                        & (catalog['Z']<=zmax) 
                     )

        catalog=catalog[select]
        
        print(len(catalog), "galaxies in redshift limits")
        
        # Cut on mask
        #The pixel IDs for every (ra, dec) position
        pxid=hp.ang2pix(nside, catalog['RA'], catalog['DEC'], nest=True,lonlat=True)
        #Select the galaxies that fall within the mask
        select = np.isin(pxid, mask[mask["DONE"]==1]["HPXPIXEL"])
        catalog=catalog[select]
        
        print(len(catalog), "galaxies within smoothed angular mask")
        
        #Quality cuts 
        #made to match Ross 2024, The Dark Energy Spectroscopic Instrument: Construction of Large-scale Structure Catalogs
        select = np.where(
                   (catalog['ZWARN']==0) 
                   & (catalog['DELTACHI2']>40) 
        )   
        
        catalog=catalog[select]
        
        print(len(catalog), "galaxies in final catalog")
        
        
        #split into NGC and SGC
        select = (catalog['RA'] < 304) * (catalog['RA'] > 83)

        print(len(catalog),'galaxies in NGC')
        print(len(catalog),'galaxies in SGC')
        
        #save catalog
        out=Table([catalog['TARGETID'],
                   catalog['RA'],
                   catalog['DEC'],
                   catalog['Z'],
                   catalog['BGS_TARGET'],
                   catalog['ABSMAG01_SDSS_U'],
                   catalog['ABSMAG01_SDSS_G'],
                   catalog['ABSMAG01_SDSS_R'],
                   catalog['ABSMAG01_SDSS_Z'],
                   catalog['LOGMSTAR'], 
                   catalog['SFR'], 
                   catalog["HALPHA_EW"]
                  ],
                   
                   names=['TARGETID','RA','DEC','Z', 'BGS_TARGET',
                          'ABSMAG01_SDSS_U','ABSMAG01_SDSS_G', 'ABSMAG01_SDSS_R', 'ABSMAG01_SDSS_Z',
                          'LOGMSTAR', 'SFR', 'HALPHA_EW']
                 )

        ngc_catalog = vstack([ngc_catalog, out[select]])
        sgc_catalog = vstack([sgc_catalog, out[~select]])

## Enter Galaxies into Void Class

In [ ]:
voidfinder_voids_path_ngc = '/global/homes/j/jkneiss/DESIVAST_NGC_VoidFinder_Output.fits'
voidfinder_voids_path_sgc = '/global/homes/j/jkneiss/DESIVAST_SGC_VoidFinder_Output.fits'
voidfinder_catalog = void_catalog.VoidFinderCatalogStacked(['N','S'], [voidfinder_voids_path_ngc, voidfinder_voids_path_sgc], edge_buffer=0)
voidfinder_catalog.add_galaxies(['tmp','tmp'], [ngc_catalog, sgc_catalog], 
                                [f'/global/homes/j/jkneiss/DR2_N_VF_vflag.fits', f'/global/homes/j/jkneiss/DR2_S_VF_vflag.fits'], 
                                redshift_name='Z', ra_name = 'RA', dec_name='DEC'
                               )

## Attatch Vflags

In [ ]:
# VoidFinder
ngc_catalog['VF_FLAG'] = voidfinder_catalog['N'].galaxies['vflag']
sgc_catalog['VF_FLAG'] = voidfinder_catalog['S'].galaxies['vflag']

#Watershed algorithms (not investigated for now)
"""galaxy_table['VV_FLAG'] = 999
galaxy_table['VV_FLAG'][select_ngc] = vide_catalog['N'].galaxies['vflag']
galaxy_table['VV_FLAG'][~select_ngc] = vide_catalog['S'].galaxies['vflag']
galaxy_table['VR_FLAG'] = 999
galaxy_table['VR_FLAG'][select_ngc] = revolver_catalog['N'].galaxies['vflag']
galaxy_table['VR_FLAG'][~select_ngc] = revolver_catalog['S'].galaxies['vflag']""";

## Cut out Survey Edges

In [ ]:
# SGC edge cut
sgc_edge_cut = voidfinder_catalog['S'].galaxy_membership(return_selector=True)

In [ ]:
# NGC edge cut
ngc_edge_cut = voidfinder_catalog['N'].galaxy_membership(return_selector=True)

## Create Final Table

In [ ]:
# construct final galaxy table for analysis, using the edge cuts to remove edge galaxies
galaxy_table = vstack([ngc_catalog[ngc_edge_cut[0] + ngc_edge_cut[1]], sgc_catalog[sgc_edge_cut[0] + sgc_edge_cut[1]]])
del ngc_catalog, sgc_catalog

In [ ]:
#Save for later

outfile = '/global/homes/j/jkneiss/galaxy_table_DR2_voids.fits'
galaxy_table.write(outfile, format='fits', overwrite=True)